# InteractSmi-Net: Drug–Target Interaction Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jinlong-abc/InteractSmi_Net/blob/master/InteractSmi_Net_Demo.ipynb)

**InteractSmi-Net** predicts protein–ligand binding affinity and generates interpretable **InteractSmi fingerprints** — a sequence-based representation of protein–ligand interaction patterns.  
No 3D structure required: the model takes only a SMILES string and a pre-computed protein ESM-C embedding.

---

### Notebook contents
| Section | Description |
|---------|-------------|
| 1 | Environment setup and model loading |
| 2 | Demo predictions on 4 benchmark compounds |
| 3 | Custom prediction — try your own SMILES |
| 4 | InteractSmi fingerprint analysis |

### Demo targets
| UniProt | Protein | Therapeutic area |
|---------|---------|------------------|
| P00749 | uPA (urokinase) | Thrombosis / oncology |
| P22756 | AANAT | Melatonin / circadian rhythm |
| P03372 | ERα (ESR1) | Hormone therapy / breast cancer |
| O14965 | Aurora-A (AURKA) | Kinase / oncology |

> **Runtime:** Set *Runtime → Change runtime type → T4 GPU* for best performance.

## Section 1: Environment Setup

In [ ]:
# Install the only non-standard dependency (PyTorch/NumPy/SciPy already in Colab)
!pip install rdkit -q

# Clone repository (includes pre-trained checkpoints and demo data)
import os
if not os.path.exists('/content/InteractSmi_Net'):
    !git clone https://github.com/jinlong-abc/InteractSmi_Net.git
else:
    print('Repository already present — skipping clone')

print('Done!')

In [ ]:
import os, sys

REPO_DIR = '/content/InteractSmi_Net'
SRC_DIR  = os.path.join(REPO_DIR, 'src')

os.chdir(SRC_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
import pickle
import numpy as np
import re
from collections import Counter
from torch.autograd import Variable

from test.Test_GraphBuilder  import MolecularGraphBuilder_TEST
from test.Test_Config        import get_config
from test.Test_model_evalue  import batch_pad, batch2tensor_test, process_protein_embeddings
from model.model             import FusionCPI

print('All modules imported successfully!')

In [ ]:
# ── Configuration (default paths are relative to src/) ────────────────────────
import sys
sys.argv = ['']   # prevent Jupyter's -f argument from conflicting with argparse
params = get_config()

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — running on CPU (inference will be slower)')

# ── Vocabulary ────────────────────────────────────────────────────────────────
with open(params.vocab, 'rb') as f:
    vocab = pickle.load(f)
print(f'Vocabulary: {len(vocab["token2id"])} tokens')

# ── Pre-computed protein ESM-C embeddings ─────────────────────────────────────
with open(params.prot_esmc_embed, 'rb') as f:
    prot_embed_dict = pickle.load(f)
print(f'Proteins in demo set: {list(prot_embed_dict.keys())}')

# ── Molecular graph builder ───────────────────────────────────────────────────
builder = MolecularGraphBuilder_TEST(
    params.fingerprint_dict_path,
    params.atom_dict_path,
    params.bond_dict_path,
)

# ── Model ─────────────────────────────────────────────────────────────────────
with open(params.fingerprint_dict_path, 'rb') as f:
    fingerprint_dict = pickle.load(f)

model = FusionCPI(len(fingerprint_dict), params, vocab)
model.to(device)

checkpoint  = torch.load(params.best_model, map_location=device, weights_only=False)
state_dict  = checkpoint.get('model_state_dict',
              checkpoint.get('model_state', checkpoint))
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded — {n_params:,} parameters')
print(f'Checkpoint  : {params.best_model}')

## Section 2: Demo Predictions

Run predictions on the four benchmark protein–ligand pairs included in the repository.

In [ ]:
def predict(
    normal_smiles_list,
    charged_smiles_list,
    uniprot_list,
    sequence_list,
    model,
    builder,
    prot_embed_dict,
    vocab,
    device,
    params,
    max_len=200,
):
    """Run binding-affinity prediction and InteractSmi generation."""
    results = []

    for norm_smi, chg_smi, uid, seq in zip(
        normal_smiles_list, charged_smiles_list, uniprot_list, sequence_list
    ):
        # Molecular graph
        compounds, adjacencies = builder.batch_mol_to_graph([norm_smi], radius=1)
        if not compounds:
            print(f'  [WARN] Could not parse SMILES: {norm_smi}')
            results.append(None)
            continue

        # Protein embedding
        prot_embed, prot_mask = process_protein_embeddings([uid], prot_embed_dict, device)

        # Tokenise inputs
        data_list = [
            np.array(compounds,   dtype=object),
            np.array(adjacencies, dtype=object),
            np.array([seq],       dtype=object),
            np.array([chg_smi],   dtype=object),
        ]
        atoms_pad, atoms_mask, adj_pad, Smiles, Smiles_mask, Seq, Seq_mask = \
            batch2tensor_test(data_list, device, vocab=vocab, params=params)

        batch_data = (
            atoms_pad, atoms_mask, adj_pad,
            prot_embed, prot_mask,
            Smiles, Smiles_mask,
            Seq, Seq_mask,
        )

        # Inference
        with torch.no_grad():
            prediction, generated = model.inference(*batch_data, max_len=max_len)

        pred_val = float(prediction.squeeze().cpu())

        # Decode InteractSmi tokens
        interactsmi = ''
        if generated is not None:
            tokens = []
            for idx in generated[0].cpu().tolist():
                tok = vocab['id2token'].get(idx, '<UNK>')
                if tok == '<EOS>':
                    break
                if tok != '<PAD>':
                    tokens.append(tok)
            interactsmi = ' '.join(tokens)

        results.append({
            'smiles'        : norm_smi,
            'uniprot'       : uid,
            'predicted_pKd' : pred_val,
            'predicted_Kd_nM': 10**(-pred_val) * 1e9,
            'interactsmi'   : interactsmi,
        })

    return results

In [ ]:
# Load demo CSV
demo_data = []
with open(params.test_data, 'r') as f:
    header = next(f)
    for line in f:
        parts = line.strip().split(',')
        demo_data.append({
            'id'            : parts[0],
            'uniprot'       : parts[1],
            'assay'         : parts[2],
            'experimental'  : float(parts[3]),
            'normal_smiles' : parts[4],
            'smiles'        : parts[5],
            'sequence'      : parts[9],
        })

print(f'Loaded {len(demo_data)} demo samples')

PROTEIN_NAMES = {
    'P00749': 'uPA',
    'P22756': 'AANAT',
    'P03372': 'ERα',
    'O14965': 'Aurora-A',
}

In [ ]:
print('Running predictions...\n')

demo_results = predict(
    [d['normal_smiles'] for d in demo_data],
    [d['smiles']        for d in demo_data],
    [d['uniprot']       for d in demo_data],
    [d['sequence']      for d in demo_data],
    model, builder, prot_embed_dict, vocab, device, params,
)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{"ID":<8} {"Protein":<12} {"Assay":<25} {"Exp. pKd":<12} {"Pred. pKd":<12} {"|Error|":<8}')
print('-' * 80)
for d, r in zip(demo_data, demo_results):
    if r is None:
        continue
    pname = PROTEIN_NAMES.get(d['uniprot'], d['uniprot'])
    err   = abs(d['experimental'] - r['predicted_pKd'])
    print(f"{d['id']:<8} {pname:<12} {d['assay']:<25} "
          f"{d['experimental']:<12.3f} {r['predicted_pKd']:<12.3f} {err:<8.3f}")

# MAE
errors = [abs(d['experimental'] - r['predicted_pKd'])
          for d, r in zip(demo_data, demo_results) if r is not None]
print('-' * 80)
print(f'Mean Absolute Error: {sum(errors)/len(errors):.3f} pKd units')

## Section 3: Custom Prediction

Change `MY_SMILES` and `MY_UNIPROT` to predict binding affinity for any molecule against one of the four demo targets.

| `MY_UNIPROT` | Protein |
|---|---|
| `'P00749'` | uPA (urokinase-type plasminogen activator) |
| `'P22756'` | AANAT (arylalkylamine N-acetyltransferase) |
| `'P03372'` | ERα / ESR1 (estrogen receptor alpha) |
| `'O14965'` | Aurora kinase A (AURKA) |

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║          CHANGE THESE TWO LINES                      ║
# ╚══════════════════════════════════════════════════════╝

MY_SMILES  = 'CC(=O)Oc1ccccc1C(=O)O'   # Aspirin (example)
MY_UNIPROT = 'P00749'                   # uPA

# ─────────────────────────────────────────────────────────

if MY_UNIPROT not in prot_embed_dict:
    print(f'Error: {MY_UNIPROT} not in demo embedding dict.')
    print(f'Available UniProt IDs: {list(prot_embed_dict.keys())}')
else:
    target = next(d for d in demo_data if d['uniprot'] == MY_UNIPROT)
    pname  = PROTEIN_NAMES.get(MY_UNIPROT, MY_UNIPROT)

    print(f'Molecule : {MY_SMILES}')
    print(f'Target   : {MY_UNIPROT} ({pname})')
    print('Predicting...\n')

    custom_results = predict(
        [MY_SMILES],
        [MY_SMILES],          # Use plain SMILES as decoder prefix
        [MY_UNIPROT],
        [target['sequence']],
        model, builder, prot_embed_dict, vocab, device, params,
    )

    if custom_results and custom_results[0] is not None:
        r = custom_results[0]
        print(f'Predicted pKd : {r["predicted_pKd"]:.3f}')
        print(f'Predicted Kd  : {r["predicted_Kd_nM"]:.2f} nM')
        print(f'\nGenerated InteractSmi:')
        print(r['interactsmi'])

## Section 4: InteractSmi Fingerprint Analysis

The **InteractSmi fingerprint** encodes protein–ligand interaction patterns as a sequence of annotated SMILES tokens.  
Each `{Residue Position Type}` tag marks an interaction between an atom and a binding-site residue:

| Tag | Interaction type |
|-----|------------------|
| `HB` | Hydrogen bond |
| `Hy` | Hydrophobic contact |
| `VDW` | van der Waals |
| `IB` | Ionic / electrostatic |

In [ ]:
INTERACTION_NAMES = {
    'HB' : 'Hydrogen bond',
    'Hy' : 'Hydrophobic',
    'VDW': 'van der Waals',
    'IB' : 'Ionic/Electrostatic',
}


def parse_interactsmi(fingerprint: str):
    """Return list of (residue, position, interaction_type) tuples."""
    return re.findall(r'\{\s*(\w+)\s+(\d+)\s+(\w+)\s*\}', fingerprint)


def display_interactsmi(result, label=''):
    fp     = result['interactsmi']
    iacts  = parse_interactsmi(fp)
    pname  = PROTEIN_NAMES.get(result['uniprot'], result['uniprot'])

    print(f'  Molecule : {label or result["smiles"][:60]}')
    print(f'  Protein  : {result["uniprot"]} ({pname})')
    print(f'  Pred. pKd: {result["predicted_pKd"]:.3f}  '
          f'(Kd ≈ {result["predicted_Kd_nM"]:.1f} nM)')

    if not iacts:
        print('  (No interaction tokens found)')
        return

    type_counts = Counter(itype for _, _, itype in iacts)
    residue_counts = Counter(f'{res}{pos}' for res, pos, _ in iacts)

    print(f'\n  Interaction summary ({len(iacts)} total tokens):')
    for itype, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
        bar = '█' * cnt
        print(f'    {INTERACTION_NAMES.get(itype, itype):<25} {bar} ({cnt})')

    print(f'\n  Key binding residues:')
    for res_pos, cnt in residue_counts.most_common(8):
        print(f'    {res_pos:<10} {cnt} contact(s)')


# ── Analyse all demo results ───────────────────────────────────────────────────
for d, r in zip(demo_data, demo_results):
    if r is None:
        continue
    print('=' * 60)
    display_interactsmi(r, label=d['id'])
    print()

In [ ]:
# Optional: visualise interaction-type distribution with matplotlib

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(demo_data), figsize=(14, 4), sharey=False)

COLORS = {'HB': '#2196F3', 'Hy': '#FF9800', 'VDW': '#4CAF50', 'IB': '#E91E63'}

for ax, (d, r) in zip(axes, zip(demo_data, demo_results)):
    if r is None:
        ax.axis('off')
        continue

    iacts  = parse_interactsmi(r['interactsmi'])
    counts = Counter(itype for _, _, itype in iacts)
    labels = [INTERACTION_NAMES.get(k, k) for k in counts]
    values = list(counts.values())
    colors = [COLORS.get(k, '#9E9E9E') for k in counts]

    ax.bar(range(len(labels)), values, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    pname = PROTEIN_NAMES.get(d['uniprot'], d['uniprot'])
    ax.set_title(f"{d['id']}\n{pname}  pKd={r['predicted_pKd']:.2f}", fontsize=9)
    ax.set_ylabel('Count')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('InteractSmi — interaction type distribution per compound', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('/content/interactsmi_interactions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to /content/interactsmi_interactions.png')

---
## Notes

### Adding new target proteins
To predict against a protein not in the demo set, pre-compute ESM-C embeddings and add them to the dict:

```python
# Pre-compute with ESM3 (requires EvolutionaryScale account)
from esm.models.esmc import ESMC
esmc = ESMC.from_pretrained('esmc_300m')
# ... embed your protein sequence ...
prot_embed_dict['YOUR_UNIPROT'] = {'embed': embed_tensor, 'mask': mask_tensor}
```

### Citation
If you use InteractSmi-Net in your research, please cite the associated paper.

### License
InteractSmi-Net is released under the [AGPL-3.0 license](https://github.com/jinlong-abc/InteractSmi_Net/blob/master/LICENSE).  
InteractSmi technology is patent-pending.